# Tarea grupal: "EnviRápido", centro de distribución de paquetes

**Entrega: AULA EC-B — Grupo 4**


**Grupos de 5 · Lenguaje: Python 

Este cuaderno contiene el **enunciado**, los **mínimos necesarios**, los **puntos de libertad**, los **entregables** y la **rúbrica**. Al final hay un espacio de trabajo con una celda de código por cada mínimo.



---
# 1. El problema
Una empresa de mensajería, **EnviRápido**, tiene un centro de distribución que hoy funciona con cuadernos y hojas de papel. Su dueño sufre estos problemas:

- Cada día llegan cientos de paquetes y se atienden **en el orden en que llegaron**, pero a veces los empleados se saltan paquetes sin darse cuenta.
- Los clientes llaman preguntando *"¿dónde está mi paquete?"* y el empleado tarda minutos en encontrarlo revisando hoja por hoja.
- A veces un empleado marca un paquete como "entregado" por error y no hay forma de corregirlo ordenadamente.
- La empresa reparte a **zonas/sucursales** (cientos de ellas, con un código numérico) y necesita saber rápidamente a cuál corresponde cada código.
- Hay paquetes que llegan **sin código de seguimiento**, solo con el nombre del cliente, y también hay que poder ubicarlos.

**Tu grupo es una empresa de software** que debe ofrecerle al dueño un prototipo que resuelva estos problemas. Ustedes deciden qué datos tiene un paquete, cómo se organiza el sistema y qué estructuras usar. El dueño no sabe de programación: tendrán que **convencerlo** de que su solución es buena y de por qué.

---
# 2. Mínimos necesarios (para aprobar)

1. **Modelado propio:** definir qué es un paquete, un cliente y una zona (clases o estructuras), y explicar por escrito qué decidieron y por qué.
2. **Recepción y atención en orden de llegada**, con posibilidad de ver quién sigue.
3. **Consulta rápida por código de seguimiento**, usando **una tabla hash implementada por ustedes** (sin usar `dict` ni `set` de Python para la estructura principal), con **manejo de colisiones**.
4. **Corrección de errores** (deshacer acciones, al menos las últimas) usando una estructura adecuada.
5. **Búsqueda de zona por código** sobre una lista ordenada, usando búsqueda binaria.
6. **Búsqueda de paquetes por nombre de cliente** (sin código), usando búsqueda secuencial.
7. **Pruebas:** un programa de ejemplo con al menos 20 paquetes que demuestre cada funcionalidad, incluyendo casos límite (cola vacía, código inexistente, colisión, deshacer sin historial).
8. **Medición:** comparar cuántas comparaciones hace cada búsqueda (secuencial vs. binaria vs. hash) con 100, 1000 y 10000 datos, y mostrar los resultados.

---
# 3. Puntos de libertad (donde tienen que pensar)
Estas decisiones **no las da el enunciado**; deben tomarlas y justificarlas:

- ¿Qué pasa con los paquetes urgentes o con prioridad? ¿Rompen la regla del orden de llegada?
- ¿Qué se hace si dos paquetes tienen el mismo nombre de cliente?
- ¿Cuál es el tamaño de la tabla hash y qué función hash usan? ¿Qué hacen cuando se llena?
- ¿Qué acciones se pueden deshacer y cuáles no?
- ¿La lista de zonas puede cambiar? Si cambia, ¿cómo se mantiene ordenada?
- ¿Cómo reportan al usuario que algo no se encontró?

**Extras opcionales (para nota adicional):** reasignar paquetes entre zonas, estadísticas del día, interfaz de menú por consola, carga de datos desde archivo.

---
# 4. Entregables y "venta"

| Entregable | Detalle |
|---|---|
| Código | Este cuaderno `.ipynb` (o archivos `.py`), comentado y ordenado |
| **Presentación de venta (10 min )** | Demostración en vivo + argumento de por qué su solución es la mejor |

**Reglas de la "venta":** los 5 integrantes deben hablar. El público (el resto de la clase y el docente) actúa como el dueño de la empresa y puede hacer preguntas como *"¿qué pasa si hay 1 millón de paquetes?"* o *"¿por qué no usaron una lista para esto?"*.



---
# ESPACIO DE TRABAJO DEL GRUPO

### Equipo y roles
Completen con los nombres reales y roten los roles al preparar la defensa.

| Integrante | Rol principal | Rol de la defensa |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |

### Registro de decisiones de diseño
- **Modelo:** Cliente tiene nombre; Zona tiene código y nombre; Paquete relaciona ambos, un código opcional, urgencia y estado. Así se representa lo necesario sin duplicar la lógica de búsqueda.
- **Urgentes:** se identifican, pero respetan la cola FIFO; nadie se salta la llegada.
- **Nombres repetidos:** la búsqueda secuencial devuelve todos los paquetes coincidentes, con o sin código, ignorando mayúsculas y espacios exteriores.
- **Hash:** suma polinómica determinista de caracteres y encadenamiento separado con listas; capacidad inicial 8, duplicada al superar factor de carga 0,75. Se reubican los elementos. Los códigos duplicados se rechazan.
- **Deshacer:** se revierten recepción y atención en orden inverso; se conserva la posición en la cola y se actualizan registro e índice. Consultas y cambios de zonas no forman parte del historial.
- **Zonas:** se pueden agregar, rechazando códigos repetidos; se inserta en la posición ordenada. La búsqueda usa binaria.
- **No encontrado:** las consultas individuales devuelven `None`, la consulta por nombre una lista vacía y deshacer `False`; la demostración presenta mensajes legibles.

### Preparación: módulos
Importen aquí lo que necesiten (por ejemplo `randint` para generar datos de prueba). **Recuerden:** no usar `dict` ni `set` para la tabla hash principal.

In [1]:
from dataclasses import dataclass
from collections import deque

### Mínimo 1 · Modelado: paquete, cliente y zona
Definan las clases o estructuras. Antes de programar, escriban en papel los **sustantivos** y los **verbos** del problema.

In [2]:
@dataclass(frozen=True)
class Cliente:
    nombre: str

@dataclass(frozen=True)
class Zona:
    codigo: int
    nombre: str

@dataclass
class Paquete:
    cliente: Cliente
    zona: Zona
    codigo: str | None = None
    urgente: bool = False
    estado: str = "recibido"

    def __post_init__(self):
        if self.codigo is not None and (not isinstance(self.codigo, str) or not self.codigo.strip()):
            raise ValueError("El código debe ser texto no vacío o None")

### Mínimo 2 · Recepción y atención en orden de llegada
Necesitan: recibir un paquete, atender el siguiente y **ver quién sigue** sin sacarlo.

In [3]:
class ColaLlegada:
    """Cola FIFO: agregar al final, consultar o retirar del frente."""
    def __init__(self):
        self._datos = deque()

    def recibir(self, paquete):
        self._datos.append(paquete)

    def siguiente(self):
        return self._datos[0] if self._datos else None

    def atender(self):
        return self._datos.popleft() if self._datos else None

    def __len__(self):
        return len(self._datos)

### Mínimo 3 · Consulta por código de seguimiento (tabla hash propia)
Implementen la tabla hash completa: función hash, insertar, buscar y manejo de colisiones. **Sin `dict` ni `set`.**

In [4]:
class TablaHash:
    """Tabla propia con encadenamiento separado; no usa dict ni set."""
    def __init__(self, capacidad=8):
        if capacidad < 1:
            raise ValueError("La capacidad debe ser positiva")
        self.tabla = [[] for _ in range(capacidad)]
        self.cantidad = 0

    def indice(self, clave):
        h = 0
        for caracter in clave:
            h = (31 * h + ord(caracter)) % len(self.tabla)
        return h

    def buscar_con_comparaciones(self, clave):
        comparaciones = 0
        for codigo, paquete in self.tabla[self.indice(clave)]:
            comparaciones += 1  # una comparación entre códigos almacenados
            if codigo == clave:
                return paquete, comparaciones
        return None, comparaciones

    def buscar(self, clave):
        return self.buscar_con_comparaciones(clave)[0]

    def insertar(self, clave, paquete):
        if not isinstance(clave, str) or not clave.strip():
            raise ValueError("El código debe ser texto no vacío")
        if self.buscar(clave) is not None:
            raise ValueError("Código de seguimiento duplicado")
        if (self.cantidad + 1) / len(self.tabla) > 0.75:
            self._redimensionar()
        self.tabla[self.indice(clave)].append((clave, paquete))
        self.cantidad += 1

    def _redimensionar(self):
        anterior = self.tabla
        self.tabla = [[] for _ in range(2 * len(anterior))]
        for cubeta in anterior:
            for codigo, paquete in cubeta:
                self.tabla[self.indice(codigo)].append((codigo, paquete))

    def eliminar(self, clave):
        cubeta = self.tabla[self.indice(clave)]
        for i, (codigo, paquete) in enumerate(cubeta):
            if codigo == clave:
                del cubeta[i]
                self.cantidad -= 1
                return paquete
        return None

### Mínimo 4 · Deshacer acciones
Decidan qué acciones se registran y **qué información hay que guardar** para poder revertirlas.

In [5]:
class CentroDistribucion:
    """Coordina cola, registro, índice y pila de acciones reversibles."""
    def __init__(self):
        self.cola = ColaLlegada()
        self.indice = TablaHash()
        self.paquetes = []
        self.historial = []  # pila: la última acción es la primera en deshacerse
        self.zonas = []

    def recibir(self, paquete):
        if paquete.estado != "recibido" or any(p is paquete for p in self.paquetes):
            raise ValueError("El paquete debe ser nuevo y estar recibido")
        if paquete.codigo is not None:
            self.indice.insertar(paquete.codigo, paquete)  # rechaza duplicados antes de modificar la cola
        self.paquetes.append(paquete)
        self.cola.recibir(paquete)
        self.historial.append(("recepcion", paquete))

    def atender(self):
        paquete = self.cola.atender()
        if paquete is not None:
            estado_anterior = paquete.estado
            paquete.estado = "entregado"
            self.historial.append(("atencion", paquete, estado_anterior))
        return paquete

    def deshacer(self):
        if not self.historial:
            return False
        accion = self.historial.pop()
        paquete = accion[1]
        if accion[0] == "atencion":
            paquete.estado = accion[2]
            self.cola._datos.appendleft(paquete)
        else:
            # Por el orden LIFO, todas las acciones posteriores ya se revirtieron.
            assert self.cola._datos[-1] is paquete and self.paquetes[-1] is paquete
            self.cola._datos.pop()
            self.paquetes.pop()
            if paquete.codigo is not None:
                self.indice.eliminar(paquete.codigo)
        return True

### Mínimo 5 · Búsqueda de zona por código (búsqueda binaria)
La lista de zonas debe estar **ordenada** antes de buscar.

In [6]:
def buscar_zona_con_comparaciones(zonas, codigo):
    izquierda, derecha, comparaciones = 0, len(zonas) - 1, 0
    while izquierda <= derecha:
        medio = (izquierda + derecha) // 2
        comparaciones += 1  # una comparación de la clave con el elemento central
        actual = zonas[medio].codigo
        if actual == codigo:
            return zonas[medio], comparaciones
        if actual < codigo:
            izquierda = medio + 1
        else:
            derecha = medio - 1
    return None, comparaciones

def buscar_zona(zonas, codigo):
    return buscar_zona_con_comparaciones(zonas, codigo)[0]

def agregar_zona(zonas, zona):
    izquierda, derecha = 0, len(zonas)
    while izquierda < derecha:
        medio = (izquierda + derecha) // 2
        if zonas[medio].codigo < zona.codigo:
            izquierda = medio + 1
        else:
            derecha = medio
    if izquierda < len(zonas) and zonas[izquierda].codigo == zona.codigo:
        raise ValueError("Código de zona duplicado")
    zonas.insert(izquierda, zona)

### Mínimo 6 · Búsqueda de paquetes por nombre de cliente (búsqueda secuencial)
Piensen qué devuelve su búsqueda si hay varios paquetes con el mismo nombre.

In [7]:
def buscar_por_cliente_con_comparaciones(paquetes, nombre):
    objetivo = nombre.strip().casefold()
    encontrados = []
    comparaciones = 0
    for paquete in paquetes:
        comparaciones += 1  # una comparación de nombres por paquete
        if paquete.cliente.nombre.strip().casefold() == objetivo:
            encontrados.append(paquete)
    return encontrados, comparaciones

def buscar_por_cliente(paquetes, nombre):
    return buscar_por_cliente_con_comparaciones(paquetes, nombre)[0]

### Mínimo 7 · Pruebas con al menos 20 paquetes
Demuestren **cada** funcionalidad. Incluyan casos límite: cola vacía, código inexistente, colisión y deshacer sin historial.

In [8]:
# Pruebas autocontenidas: las aserciones detienen la ejecución si algo falla.
centro = CentroDistribucion()
assert centro.cola.siguiente() is None and centro.atender() is None
assert centro.deshacer() is False
print("Cola vacía y deshacer sin historial: correctos")
for codigo in [30, 10, 20]:
    agregar_zona(centro.zonas, Zona(codigo, f"Zona {codigo}"))
assert [z.codigo for z in centro.zonas] == [10, 20, 30]
assert buscar_zona(centro.zonas, 20).nombre == "Zona 20"
assert buscar_zona(centro.zonas, 99) is None
try:
    agregar_zona(centro.zonas, Zona(20, "Duplicada"))
    raise AssertionError("Se aceptó una zona duplicada")
except ValueError:
    pass
zona = centro.zonas[0]
for i in range(20):
    cliente = Cliente("Ana" if i in (0, 1) else f"Cliente {i}")
    centro.recibir(Paquete(cliente, zona, None if i == 1 else f"ENV-{i:03d}", urgente=(i == 5)))
assert len(centro.paquetes) == len(centro.cola) == 20
assert centro.indice.buscar("ENV-000") is centro.paquetes[0]
assert centro.indice.buscar("inexistente") is None
assert len(buscar_por_cliente(centro.paquetes, " ana ")) == 2
assert buscar_por_cliente(centro.paquetes, "Nadie") == []
primero = centro.cola.siguiente()
assert centro.atender() is primero and primero.estado == "entregado"
assert centro.cola.siguiente() is centro.paquetes[1]
assert centro.deshacer() and primero.estado == "recibido" and centro.cola.siguiente() is primero
# Deshacer una recepción actualiza la cola, el registro y la tabla.
ultimo = Paquete(Cliente("Temporal"), zona, "TEMP")
centro.recibir(ultimo)
assert centro.deshacer() and centro.indice.buscar("TEMP") is None
assert len(centro.paquetes) == len(centro.cola) == 20
try:
    centro.recibir(Paquete(Cliente("Duplicado"), zona, "ENV-000"))
    raise AssertionError("Se aceptó un código duplicado")
except ValueError:
    pass
assert len(centro.paquetes) == len(centro.cola) == 20
# Colisión deliberada: A y I caen en la misma cubeta con capacidad 8.
tabla = TablaHash(8)
a, b = Paquete(Cliente("A"), zona, "A"), Paquete(Cliente("B"), zona, "I")
assert tabla.indice("A") == tabla.indice("I")
tabla.insertar("A", a); tabla.insertar("I", b)
assert tabla.buscar("A") is a and tabla.buscar("I") is b
assert tabla.buscar_con_comparaciones("I")[1] == 2
assert tabla.eliminar("A") is a and tabla.buscar("I") is b
# Crecimiento y preservación de todos los elementos.
for i in range(40):
    tabla.insertar(f"C{i}", Paquete(Cliente(str(i)), zona, f"C{i}"))
assert len(tabla.tabla) > 8 and all(tabla.buscar(f"C{i}") is not None for i in range(40))
print("20 paquetes: FIFO, consulta, nombres repetidos, zonas y deshacer: correctos")
print("Código inexistente, duplicados, colisión, eliminación y redimensionamiento: correctos")
print("Siguiente:", centro.cola.siguiente().cliente.nombre)

Cola vacía y deshacer sin historial: correctos
20 paquetes: FIFO, consulta, nombres repetidos, zonas y deshacer: correctos
Código inexistente, duplicados, colisión, eliminación y redimensionamiento: correctos
Siguiente: Ana


### Mínimo 8 · Medición de comparaciones
Midan cuántas comparaciones hace cada búsqueda (secuencial, binaria y hash) con **100, 1000 y 10000** datos y muestren una tabla con los resultados. Después escriban en la celda siguiente qué observan.

In [9]:
# Una comparación = confrontar la clave buscada con la clave de un elemento.
# Se busca una clave existente al final; secuencial busca TODOS los nombres iguales.
print(f"{'Datos':>8} {'Secuencial':>12} {'Binaria':>10} {'Hash':>8}")
resultados = []
for n_datos in (100, 1000, 10000):
    zonas = [Zona(i, f"Zona {i}") for i in range(n_datos)]
    paquetes = [Paquete(Cliente(f"Cliente {i}"), zonas[i], f"P{i:05d}") for i in range(n_datos)]
    tabla = TablaHash()
    for paquete in paquetes:
        tabla.insertar(paquete.codigo, paquete)
    encontrados, sec = buscar_por_cliente_con_comparaciones(paquetes, f"Cliente {n_datos - 1}")
    zona_encontrada, binaria = buscar_zona_con_comparaciones(zonas, n_datos - 1)
    paquete_encontrado, hashed = tabla.buscar_con_comparaciones(f"P{n_datos - 1:05d}")
    assert len(encontrados) == 1 and zona_encontrada is zonas[-1] and paquete_encontrado is paquetes[-1]
    resultados.append((n_datos, sec, binaria, hashed))
    print(f"{n_datos:>8} {sec:>12} {binaria:>10} {hashed:>8}")

   Datos   Secuencial    Binaria     Hash
     100          100          7        2
    1000         1000         10        3
   10000        10000         14        4


### Análisis de las mediciones
La tabla muestra una búsqueda exitosa del último elemento para cada tamaño; los contadores registran una comparación de clave por elemento examinado. Son operaciones distintas: la búsqueda por nombre recorre todos los paquetes para encontrar duplicados, mientras las otras devuelven un único código. No se cuenta el costo de calcular el hash ni de construir las estructuras.

Al multiplicar los datos por 10, la secuencial multiplica sus comparaciones por 10: crece linealmente, O(n). La binaria crece aproximadamente en 3 o 4 comparaciones: O(log n), siempre que las zonas estén ordenadas. El hash depende de las colisiones; suele requerir pocas comparaciones en promedio con una distribución adecuada y factor de carga controlado, pero puede llegar a O(n) en el peor caso. Los valores concretos de esta única consulta no demuestran el rendimiento promedio ni el tiempo real. Con un millón de paquetes también serían relevantes la memoria, la persistencia en disco, la concurrencia y pruebas con muchas consultas; este prototipo guarda todo en memoria.

### Lista de verificación antes de entregar
- [x] Los 8 mínimos funcionan y están demostrados.
- [x] La tabla hash no usa `dict` ni `set` y maneja colisiones.
- [x] El código está comentado y ordenado.
- [ ] Completar nombres y roles; los 5 integrantes deben poder explicar cualquier parte.

### Uso interactivo para el empleado
Ejecute todas las celdas anteriores y después esta sección. Ejecute la última celda para abrir el menú; escriba el número de la opción y responda a las preguntas. Primero agregue al menos una zona y luego registre paquetes. La opción 9 termina la sesión. Los datos se conservan solo en memoria mientras la sesión esté activa; al reiniciar el menú se crea un centro vacío. Las pruebas anteriores usan sus propios datos y no cargan paquetes en el menú.


In [ ]:
def iniciar_menu():
    """Interfaz para operar el centro durante la sesión del cuaderno."""
    centro = CentroDistribucion()
    def mostrar(paquete):
        if paquete is None:
            print("No se encontró ningún paquete.")
        else:
            print(f"Cliente: {paquete.cliente.nombre} | Código: {paquete.codigo or 'sin código'} | "
                  f"Zona: {paquete.zona.nombre} ({paquete.zona.codigo}) | Estado: {paquete.estado} | "
                  f"Urgente: {'sí' if paquete.urgente else 'no'}")
    while True:
        print("\nENVI RÁPIDO\n1. Agregar zona\n2. Recibir paquete\n3. Ver siguiente\n"
              "4. Atender siguiente\n5. Buscar por código\n6. Buscar por cliente\n"
              "7. Buscar zona\n8. Deshacer última acción\n9. Salir")
        opcion = input("Seleccione una opción: ").strip()
        try:
            if opcion == "1":
                codigo = int(input("Código numérico de la zona: "))
                nombre = input("Nombre de la zona: ").strip()
                if not nombre:
                    raise ValueError("El nombre no puede estar vacío")
                agregar_zona(centro.zonas, Zona(codigo, nombre))
                print("Zona agregada.")
            elif opcion == "2":
                nombre = input("Nombre del cliente: ").strip()
                if not nombre:
                    raise ValueError("El nombre no puede estar vacío")
                codigo_zona = int(input("Código de la zona: "))
                zona = buscar_zona(centro.zonas, codigo_zona)
                if zona is None:
                    print("Zona no encontrada. Agréguela primero.")
                    continue
                codigo = input("Código de seguimiento (vacío si no tiene): ").strip() or None
                urgente = input("¿Es urgente? (s/n): ").strip().lower()
                if urgente not in ("s", "n"):
                    raise ValueError("Responda s o n")
                centro.recibir(Paquete(Cliente(nombre), zona, codigo, urgente == "s"))
                print("Paquete recibido. Los urgentes también respetan el orden de llegada.")
            elif opcion == "3":
                mostrar(centro.cola.siguiente())
            elif opcion == "4":
                paquete = centro.atender()
                if paquete is None:
                    print("No hay paquetes pendientes.")
                else:
                    print("Paquete atendido:")
                    mostrar(paquete)
            elif opcion == "5":
                mostrar(centro.indice.buscar(input("Código de seguimiento: ").strip()))
            elif opcion == "6":
                resultados = buscar_por_cliente(centro.paquetes, input("Nombre del cliente: "))
                if not resultados:
                    print("No se encontraron paquetes para ese cliente.")
                for paquete in resultados:
                    mostrar(paquete)
            elif opcion == "7":
                zona = buscar_zona(centro.zonas, int(input("Código de la zona: ")))
                print(f"Zona: {zona.nombre} ({zona.codigo})" if zona else "Zona no encontrada.")
            elif opcion == "8":
                print("Última recepción o atención deshecha." if centro.deshacer() else "No hay acciones para deshacer.")
            elif opcion == "9":
                print("Sesión finalizada.")
                return centro
            else:
                print("Opción inválida; elija un número del 1 al 9.")
        except ValueError as error:
            print(f"Dato inválido: {error}")


In [ ]:
centro_usuario = iniciar_menu()
